In [ ]:
import numpy as np
import pandas as pd
import gc, json, time
from pathlib import Path
from scipy.ndimage import median_filter
from scipy.signal import find_peaks
import zstandard as zstd

CFG = {
    # Preprocessing settings
    'input_dir':  Path('./Test signals from previous years/Suction data/Original'),
    'output_dir': Path('./Test signals from previous years/Suction data/Preprocessed'),
    'original_fs': 50_000,     # Hz
    'target_fs':   500,        # Hz
    'adc_column':  'adc2',
    'adc_col_index': None,
    'chunk_rows': 10_000_000,
    'baseline_sec': 5.0,
    'uint16_offset': 32_768,
    'overwrite': False,
}
CFG['pool_block'] = CFG['original_fs'] // CFG['target_fs']
CFG['output_dir'].mkdir(parents=True, exist_ok=True)

In [ ]:
def sniff_csv(path):
    """Return (delimiter, has_header, n_columns) from the first line."""
    with open(path, 'r') as f:
        first = f.readline()
    delim = '\t' if first.count('\t') >= first.count(',') else ','
    parts = [p.strip() for p in first.rstrip('\r\n').split(delim) if p != '']
    is_numeric = all(p.lstrip('-').replace('.', '', 1).isdigit() for p in parts)
    return delim, (not is_numeric), len(parts)

def load_adc_chunks(path, cfg):
    """Yield the ADC column as int16 arrays, one chunk at a time."""
    delim, header, ncols = sniff_csv(path)
    if header:
        cols = pd.read_csv(path, sep=delim, nrows=0).columns.tolist()
        usecols = cfg['adc_column'] if cfg['adc_column'] in cols else cols[cfg['adc_col_index']]
    else:
        usecols = cfg['adc_col_index']

    for chunk in pd.read_csv(path, sep=delim, usecols=[usecols],
                             header=0 if header else None,
                             chunksize=cfg['chunk_rows'],
                             dtype=np.int16, engine='c'):
        yield chunk.iloc[:, 0].to_numpy(dtype=np.int16)
        del chunk

In [ ]:
# Cell 3: Block-mean downsampling (no cross-chunk state, no edge artifacts)
def pool_downsample(chunks, block):
    """Mean-pool every `block` raw samples → 1 output sample."""
    parts, carry = [], np.empty(0, dtype=np.int16)
    total_raw = 0
    for arr in chunks:
        total_raw += len(arr)
        arr = np.concatenate([carry, arr]) if len(carry) else arr
        n_blocks = len(arr) // block
        if n_blocks:
            usable = n_blocks * block
            parts.append(arr[:usable].reshape(n_blocks, block).mean(axis=1, dtype=np.int16))
            carry = arr[usable:].copy()
        else:
            carry = arr
        del arr
    if len(carry):
        parts.append(np.array([carry.mean()], dtype=np.int16))
    return np.concatenate(parts), total_raw

In [ ]:
# Cell 4: Full single-file preprocessing (decimate → baseline → uint16 .bin + sidecar)
def preprocess_file(csv_path, cfg):
    bin_path  = cfg['output_dir'] / (csv_path.stem + '.bin')
    meta_path = cfg['output_dir'] / (csv_path.stem + '.meta.json')

    if bin_path.exists() and not cfg['overwrite']:
        print(f"{csv_path.name}: .bin already exists (set overwrite=True to redo)")
        return bin_path

    t0 = time.time()
    raw_mb = csv_path.stat().st_size / 1024**2

    # 1. Read + downsample
    pooled, n_raw = pool_downsample(load_adc_chunks(csv_path, cfg), cfg['pool_block'])
    expected = n_raw / cfg['pool_block']
    assert abs(len(pooled) - expected) <= 1, f"samples lost: {len(pooled)} vs {expected:.1f}"

    # 2. Baseline removal
    win = int(cfg['baseline_sec'] * cfg['target_fs'])
    win += 1 if win % 2 == 0 else 0
    centered = pooled - median_filter(pooled, size=win)

    # 3. Convert to uint16
    u16 = np.clip(centered + cfg['uint16_offset'], 0, 65_535).astype(np.uint16)
    u16.tofile(bin_path)

    # Sidecar info
    meta = {
        'source_file': csv_path.name,
        'original_size_mb': round(raw_mb, 2),
        'raw_samples': n_raw,
        'raw_fs': cfg['original_fs'],
        'fs': cfg['target_fs'],
        'samples': int(len(u16)),
        'duration_sec': round(len(u16) / cfg['target_fs'], 2),
        'dtype': 'uint16',
        'value_offset': cfg['uint16_offset'],
        'baseline_window_sec': cfg['baseline_sec'],
        'preprocess_time_sec': round(time.time() - t0, 1),
    }
    meta_path.write_text(json.dumps(meta, indent=2))

    print(f"✅ {csv_path.name}: {n_raw:,} raw → {len(u16):,} samples"
          f"({meta['duration_sec']}s) | {raw_mb:.0f} MB → {len(u16)*2/1024**2:.1f} MB"
          f"in {meta['preprocess_time_sec']}s")
    return bin_path

In [ ]:
# Cell 5: Batch run, smallest files first
files = sorted(CFG['input_dir'].glob('*.csv'), key=lambda p: p.stat().st_size)
print(f"{len(files)} files, smallest → largest:")
for f in files:
    print(f"   {f.stat().st_size/1024**2:8.1f} MB  {f.name}")
print()

for csv_path in files:
    try:
        preprocess_file(csv_path, CFG)
        gc.collect()
    except Exception as e:
        print(f"{csv_path.name}: {e}")

In [ ]:
# Cell 6: Region Detection Configuration
REGION_CONFIG = {
    'data_dir': Path('./Test signals from previous years/Suction data/Preprocessed'),
    'result_output_dir': Path('./Test signals from previous years/Suction data/Results'),

    # Activity detection thresholds
    'activity_window_samples': 50,      # 100ms sliding window at 500Hz
    'activity_multiplier': 2.0,         # std > 2× baseline = active
    'min_region_duration_sec': 0.5,     # Ignore regions shorter than this (noise)
    'padding_sec': 0.2,                 # Add 200ms buffer before/after each region
    
    # Output settings
    'compress_output': True,
    'zstd_level': 3,
    'fs': 500,
}

REGION_CONFIG['padding_samples'] = int(REGION_CONFIG['padding_sec'] * REGION_CONFIG['fs'])
REGION_CONFIG['output_dir'] = REGION_CONFIG['result_output_dir']
REGION_CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

print("✅ Region Detection Configuration")
print(f"   Activity threshold: {REGION_CONFIG['activity_multiplier']}× baseline std")
print(f"   Min region duration: {REGION_CONFIG['min_region_duration_sec']}s")
print(f"   Padding: ±{REGION_CONFIG['padding_sec']*1000:.0f} ms")
print(f"   Compress: {REGION_CONFIG['compress_output']} (zstd level {REGION_CONFIG['zstd_level'] if HAS_ZSTD else 'N/A'})")

In [ ]:
# Cell 7: Active Region Detection + Extraction
def detect_regions_and_extract(bin_path, cfg):
    """
    Detect contiguous active regions → store as blocks → compress
    Better for bursty/clustering patterns than per-peak extraction
    """
    meta_path = bin_path.with_suffix('.meta.json')
    if not meta_path.exists():
        raise FileNotFoundError(f"No metadata found for {bin_path}")

    meta = json.loads(meta_path.read_text())

    # Memory-map original data
    raw_data = np.memmap(str(bin_path), dtype='uint16', mode='r')
    centered = (raw_data.astype(np.int32) - meta['value_offset'])

    # Compute rolling std to detect activity (100ms windows)
    activity_std = pd.Series(centered).rolling(cfg['activity_window_samples'], center=True, min_periods=1).std().to_numpy()
    
    # Baseline is overall std
    baseline_std = np.std(centered)
    activity_threshold = cfg['activity_multiplier'] * baseline_std
    
    # Mark active samples (where local std > threshold)
    active_mask = activity_std > activity_threshold

    # Find contiguous active regions
    regions = []
    in_region = False
    region_start = 0
    
    for i, is_active in enumerate(active_mask):
        if is_active and not in_region:
            region_start = i
            in_region = True
        elif not is_active and in_region:
            region_end = i
            
            # Apply padding
            padded_start = max(0, region_start - cfg['padding_samples'])
            padded_end = min(len(centered), region_end + cfg['padding_samples'])
            
            # Skip too-short regions
            duration_sec = (padded_end - padded_start) / cfg['fs']
            if duration_sec >= cfg['min_region_duration_sec']:
                regions.append({
                    'global_start': int(padded_start),
                    'global_end': int(padded_end),
                    'duration_sec': round(duration_sec, 4),
                })
            
            in_region = False

    if in_region:
        padded_start = max(0, region_start - cfg['padding_samples'])
        padded_end = len(centered)
        duration_sec = (padded_end - padded_start) / cfg['fs']
        
        if duration_sec >= cfg['min_region_duration_sec']:
            regions.append({
                'global_start': int(padded_start),
                'global_end': int(padded_end),
                'duration_sec': round(duration_sec, 4),
            })

    # Concatenate all region data
    all_region_data = []
    for region in regions:
        segment = centered[region['global_start']:region['global_end']]
        all_region_data.append(segment)

    if len(all_region_data) > 0:
        compressed_regions = np.concatenate(all_region_data).astype(np.uint16) + meta['value_offset']
    else:
        compressed_regions = np.array([], dtype=np.uint16)

    # Calculate stats
    total_stored_samples = len(compressed_regions)
    total_original_samples = meta['samples']
    compression_ratio = total_original_samples / total_stored_samples if total_stored_samples > 0 else 1.0
    
    total_region_duration = sum(r['duration_sec'] for r in regions)

    # Build output metadata
    output_meta = {
        'source_file': bin_path.name,
        'original_duration_sec': meta['duration_sec'],
        'original_samples': total_original_samples,
        'num_regions': len(regions),
        'stored_regions': regions,
        'total_stored_samples': total_stored_samples,
        'compression_ratio': round(compression_ratio, 2),
        'original_size_mb': meta.get('original_size_mb', 0),
        'stored_size_mb': round(total_stored_samples * 2 / 1024**2, 3),
        'total_region_duration_sec': round(total_region_duration, 2),
        'region_efficiency': round(total_region_duration / meta['duration_sec'], 4),  # % of time stored
        'baseline_std': round(baseline_std, 2),
        'activity_threshold': round(activity_threshold, 2),
        'value_offset': meta['value_offset'],
        'fs': cfg['fs'],
        'zstd_compressed': cfg['compress_output'],
    }

    # Determine output path
    ext = '.bin.zst' if cfg['compress_output'] and HAS_ZSTD else '.bin'
    output_path = cfg['output_dir'] / (bin_path.stem + '_regions' + ext)
    meta_path_out = cfg['output_dir'] / (bin_path.stem + '_regions.meta.json')

    # Write data
    if cfg['compress_output'] and HAS_ZSTD and len(compressed_regions) > 0:
        cctx = zstd.ZstdCompressor(level=cfg['zstd_level'])
        raw_bytes = compressed_regions.tobytes()
        compressed_bytes = cctx.compress(raw_bytes)
        output_path.write_bytes(compressed_bytes)
        output_meta['stored_size_mb'] = round(len(compressed_bytes) / 1024**2, 3)
    elif len(compressed_regions) > 0:
        compressed_regions.tofile(output_path)

    # Write metadata
    meta_path_out.write_text(json.dumps(output_meta, indent=2))

    print(f"✅ {bin_path.name}: {meta['duration_sec']:.1f}s → {total_region_duration:.1f}s "
          f"({output_meta['compression_ratio']:.1f}x reduction, {output_meta['region_efficiency']*100:.1f}% stored)")

    return output_meta

In [ ]:
# Cell 8: Full Batch Region Extraction
def run_batch_region_extraction():
    """Process all .bin files with region-based extraction + compression"""
    bin_files = sorted(REGION_CONFIG['data_dir'].glob('*.bin'))

    print("\n" + "="*60)
    print("🎯 REGION-BASED EXTRACTION + COMPRESSION")
    print("="*60)

    results = []
    total_original_mb = 0
    total_stored_mb = 0
    total_time = 0

    for i, bin_path in enumerate(bin_files):
        start = time.time()

        try:
            result = detect_regions_and_extract(bin_path, REGION_CONFIG)
            elapsed = time.time() - start
            total_time += elapsed

            total_original_mb += result['original_size_mb']
            total_stored_mb += result['stored_size_mb']

            status = "✓" if elapsed < 5.0 else "⚠️ SLOW"
            print(f"[{i+1}/{len(bin_files)}] {status} {result['stored_size_mb']:.2f} MB "
                  f"in {elapsed*1000:.1f} ms")

            results.append({
                'file': bin_path.name,
                **result,
                'processing_ms': round(elapsed * 1000, 1),
            })

        except Exception as e:
            print(f"[{i+1}/{len(bin_files)}] ❌ {bin_path.name}: {e}")
            results.append({'file': bin_path.name, 'error': str(e)})

    # Summary
    print("\n" + "="*60)
    print("📊 OVERALL SUMMARY")
    print("="*60)

    total_ratio = total_original_mb / total_stored_mb if total_stored_mb > 0 else 1
    print(f"   Original: {total_original_mb:.2f} MB")
    print(f"   Optimized: {total_stored_mb:.2f} MB")
    print(f"   Total compression: {total_ratio:.1f}x")
    print(f"   Files processed: {len(results)}")
    print(f"   Total time: {total_time:.1f}s")

    # Save results
    df = pd.DataFrame(results)
    df.to_csv(REGION_CONFIG['output_dir'] / 'region_extraction_results.csv', index=False)

    return results

In [ ]:
# Cell 9: Load Region Data Utility
def load_region_data(region_bin_path):
    """Load region-based data (handles both .bin and .bin.zst)"""
    meta_path = region_bin_path.with_suffix('.meta.json')
    meta = json.loads(meta_path.read_text())

    # Load region data
    if region_bin_path.suffix == '.zst':
        if not HAS_ZSTD:
            raise ImportError("zstandard package required for .zst files")
        compressed = region_bin_path.read_bytes()
        dctx = zstd.ZstdDecompressor()
        raw_bytes = dctx.decompress(compressed)
        regions_data = np.frombuffer(raw_bytes, dtype=np.uint16)
    else:
        regions_data = np.fromfile(region_bin_path, dtype=np.uint16)

    # Convert back to centered values
    centered = regions_data.astype(np.int32) - meta['value_offset']

    # Reconstruct individual regions
    regions = []
    idx = 0
    for region_info in meta['stored_regions']:
        region_size = region_info['global_end'] - region_info['global_start']
        region_data = centered[idx:idx + region_size]
        regions.append({
            'timestamp_sec': region_info['global_start'] / meta['fs'],
            'data': region_data,
            'duration_sec': region_info['duration_sec'],
        })
        idx += region_size

    return {
        'regions': regions,
        'num_regions': len(regions),
        'meta': meta,
    }

In [ ]:
# Cell 10: RUN THE COMPLETE PIPELINE
if __name__ == "__main__":
    print("="*60)
    print("🚀 RUNNING COMPLETE PIPELINE")
    print("="*60)

    # Step 1: Preprocess CSVs to .bin (skip if already done)
    print("\nStep 1: Preprocessing...")
    files = sorted(CFG['input_dir'].glob('*.csv'), key=lambda p: p.stat().st_size)
    preprocessed_count = 0
    skipped_count = 0
    
    for csv_path in files:
        bin_path = CFG['output_dir'] / (csv_path.stem + '.bin')
        if bin_path.exists() and not CFG['overwrite']:
            skipped_count += 1
            continue
        try:
            preprocess_file(csv_path, CFG)
            preprocessed_count += 1
            gc.collect()
        except Exception as e:
            print(f"❌ {csv_path.name}: {e}")
    
    print(f"   Processed: {preprocessed_count}, Skipped: {skipped_count}")

    # Step 2: Region extraction + compression
    print("\nStep 2: Region extraction + compression...")
    results = run_batch_region_extraction()

    print("\n" + "="*60)
    print("✅ ALL DONE!")
    print("="*60)